In [23]:
# Krok 0: konfiguracja Windows pod Hugging Face cache

# Importujemy modul os do pracy ze zmiennymi srodowiskowymi.
import os

# Ustawiamy flage, aby ukryc ostrzezenie o symlinkach na Windows.
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# Ustawiamy domyslny katalog cache dla biblioteki Hugging Face.
os.environ.setdefault("HF_HOME", "../artifacts/hf_cache")

# Wypisujemy wartosc flagi, aby potwierdzic konfiguracje.
print("HF_HUB_DISABLE_SYMLINKS_WARNING=", os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"])
# Wypisujemy sciezke cache, aby potwierdzic konfiguracje.
print("HF_HOME=", os.environ["HF_HOME"])


HF_HUB_DISABLE_SYMLINKS_WARNING= 1
HF_HOME= ../artifacts/hf_cache


# 02 Dane i Tokenizacja

Cel: pobrac dane, zrobic split train/validation/test i przygotowac tokenizacje dla teacher i student.


In [24]:
# Krok 1: importujemy biblioteki do danych i tokenizacji

# Importujemy modul os do operacji na katalogach i sciezkach.
import os
# Importujemy pandas do wczytywania i obrobki tabel.
import pandas as pd
# Importujemy klasy Dataset i DatasetDict z biblioteki datasets.
from datasets import Dataset, DatasetDict
# Importujemy tokenizer z Transformers.
from transformers import AutoTokenizer

# Krok 2: konfigurujemy URL danych i modele

# Zapisujemy adres URL pliku z danymi.
DATA_URL = "https://raw.githubusercontent.com/isaaccs/sentiment-analysis-for-financial-news/master/all-data.csv"
# Ustalamy sciezke zapisu surowej wersji danych.
RAW_SAVE_PATH = "../data/all-data_raw.csv"
# Ustalamy sciezke zapisu wyczyszczonej wersji danych.
CLEAN_SAVE_PATH = "../data/all-data_clean.csv"
# Ustalamy nazwe modelu teacher.
TEACHER_MODEL_NAME = "distilroberta-base"
# Ustalamy nazwe modelu student.
STUDENT_MODEL_NAME = "prajjwal1/bert-tiny"
# Ustalamy maksymalna dlugosc sekwencji dla tokenizacji.
MAX_LENGTH = 128

# Krok 3: pobieramy dane z URL

# Wczytujemy CSV z kodowaniem latin-1 i bez naglowka.
df = pd.read_csv(DATA_URL, encoding="latin-1", header=None)

# Krok 4: normalizujemy format kolumn

# Sprawdzamy, czy plik ma tylko jedna kolumne.
if df.shape[1] == 1:
    # Rozdzielamy jedna kolumne po znaku @ na dwie kolumny.
    split_df = df.iloc[:, 0].astype(str).str.split("@", n=1, expand=True)
    # Nadajemy oczekiwane nazwy kolumn po rozdzieleniu.
    split_df.columns = ["label_text", "sentence"]
    # Podmieniamy glowna ramke danych na rozdzielona wersje.
    df = split_df
# Jesli sa co najmniej dwie kolumny, przechodzimy do alternatywnej sciezki.
else:
    # Bierzemy pierwsze dwie kolumny i tworzymy ich kopie.
    df = df.iloc[:, :2].copy()
    # Tymczasowo nazywamy kolumny jako col0 i col1.
    df.columns = ["col0", "col1"]
    # Definiujemy dozwolone etykiety sentymentu.
    label_values = {"negative", "neutral", "positive"}
    # Liczymy, jaki procent wartosci w col0 wyglada jak etykieta.
    col0_labels = df["col0"].astype(str).str.strip().str.lower().isin(label_values).mean()
    # Liczymy, jaki procent wartosci w col1 wyglada jak etykieta.
    col1_labels = df["col1"].astype(str).str.strip().str.lower().isin(label_values).mean()
    # Sprawdzamy, czy col1 bardziej przypomina kolumne etykiet.
    if col1_labels > col0_labels:
        # Zmieniamy nazwy tak, aby col1 byla etykieta, a col0 tekstem.
        df = df.rename(columns={"col1": "label_text", "col0": "sentence"})
    # W przeciwnym razie col0 traktujemy jako etykiete.
    else:
        # Zmieniamy nazwy tak, aby col0 byla etykieta, a col1 tekstem.
        df = df.rename(columns={"col0": "label_text", "col1": "sentence"})

# Krok 5: czyscimy dane i mapujemy label na int

# Normalizujemy tekst etykiety: string, trim i male litery.
df["label_text"] = df["label_text"].astype(str).str.strip().str.lower()
# Normalizujemy kolumne sentence: string i trim.
df["sentence"] = df["sentence"].astype(str).str.strip()
# Usuwamy rekordy z pustym tekstem.
df = df[df["sentence"].str.len() > 0].copy()
# Tworzymy mapowanie etykiet tekstowych na liczby klas.
label_map = {"negative": 0, "neutral": 1, "positive": 2}
# Zamieniamy etykiete tekstowa na etykiete liczbowa.
df["label"] = df["label_text"].map(label_map)
# Usuwamy rekordy, dla ktorych nie udalo sie zmapowac etykiety.
df = df.dropna(subset=["label"]).copy()
# Rzutujemy etykiete na typ int.
df["label"] = df["label"].astype(int)

# Krok 6: zapisujemy dane na dysk

# Tworzymy katalog data, jesli jeszcze nie istnieje.
os.makedirs("../data", exist_ok=True)
# Zapisujemy surowe dane z etykieta tekstowa.
df[["label_text", "sentence"]].to_csv(RAW_SAVE_PATH, index=False, encoding="utf-8")
# Zapisujemy dane treningowe z etykieta liczbowa.
df[["sentence", "label"]].to_csv(CLEAN_SAVE_PATH, index=False, encoding="utf-8")
# Wypisujemy sciezke zapisu surowych danych.
print("Zapisano:", RAW_SAVE_PATH)
# Wypisujemy sciezke zapisu wyczyszczonych danych.
print("Zapisano:", CLEAN_SAVE_PATH)
# Wypisujemy liczbe rekordow po czyszczeniu.
print("Liczba rekordow:", len(df))
# Pokazujemy 3 przykladowe rekordy do szybkiej kontroli.
print(df[["sentence", "label"]].head(3))

# Krok 7: zamieniamy pandas -> Hugging Face Dataset i robimy split 80/10/10

# Tworzymy obiekt Dataset z kolumn sentence i label.
hf_dataset = Dataset.from_pandas(df[["sentence", "label"]], preserve_index=False)
# Dzielimy dane na train (80%) i tymczasowy test (20%).
split_1 = hf_dataset.train_test_split(test_size=0.2, seed=42)
# Dzielimy tymczasowy test na validation (10%) i test (10%).
split_2 = split_1["test"].train_test_split(test_size=0.5, seed=42)
# Budujemy slownik splitow train/validation/test.
dataset = DatasetDict({
    # Ustawiamy split treningowy.
    "train": split_1["train"],
    # Ustawiamy split walidacyjny.
    "validation": split_2["train"],
    # Ustawiamy split testowy.
    "test": split_2["test"],
})

# Krok 8: dodajemy indeks rekordu, aby latwo laczyc teacher logits

# Dodajemy kolumne idx na podstawie indeksu rekordu w danym splicie.
dataset = dataset.map(lambda x, idx: {"idx": idx}, with_indices=True)
# Wypisujemy podsumowanie splitow po dodaniu idx.
print(dataset)


Zapisano: ../data/all-data_raw.csv
Zapisano: ../data/all-data_clean.csv
Liczba rekordow: 4846
                                            sentence  label
0  According to Gran , the company has no plans t...      1
1  Technopolis plans to develop in stages an area...      1
2  The international electronic industry company ...      0


Map:   0%|          | 0/3876 [00:00<?, ? examples/s]

Map:   0%|          | 0/485 [00:00<?, ? examples/s]

Map:   0%|          | 0/485 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 3876
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 485
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 485
    })
})


In [25]:
# Krok 4.5: analizujemy dlugosci tokenow, aby swiadomie dobrac MAX_LENGTH

# Importujemy numpy do liczenia percentyli i srednich.
import numpy as np

# Wczytujemy tokenizer teachera do analizy dlugosci tokenow.
teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_NAME)
# Wczytujemy tokenizer studenta do analizy dlugosci tokenow.
student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL_NAME)

# Pobieramy liste tekstow z przygotowanej ramki danych.
texts = df["sentence"].tolist()

# Definiujemy funkcje, ktora liczy dlugosc tokenow dla kazdego tekstu.
def get_token_lengths(tokenizer, text_list):
    # Tworzymy pusta liste na wyniki.
    lengths = []
    # Iterujemy po kazdym tekscie.
    for text in text_list:
        # Kodujemy tekst bez przycinania, aby poznac prawdziwa dlugosc.
        token_ids = tokenizer.encode(text, add_special_tokens=True, truncation=False)
        # Zapisujemy liczbe tokenow dla tego tekstu.
        lengths.append(len(token_ids))
    # Zwracamy liste dlugosci tokenow.
    return lengths

# Liczymy dlugosci tokenow dla teachera.
teacher_lengths = get_token_lengths(teacher_tokenizer, texts)
# Liczymy dlugosci tokenow dla studenta.
student_lengths = get_token_lengths(student_tokenizer, texts)

# Definiujemy funkcje do czytelnego podsumowania rozkladu dlugosci.
def describe_lengths(name, lengths):
    # Zamieniamy liste na tablice numpy.
    arr = np.array(lengths)
    # Wypisujemy nazwe analizowanego tokenizera.
    print(f"\n{name} token lengths:")
    # Wypisujemy podstawowe statystyki dlugosci.
    print("min=", int(arr.min()), "mean=", round(float(arr.mean()), 2), "max=", int(arr.max()))
    # Wypisujemy percentyle, ktore pomagaja dobrac max_length.
    print("p90=", int(np.percentile(arr, 90)), "p95=", int(np.percentile(arr, 95)), "p99=", int(np.percentile(arr, 99)))

# Pokazujemy statystyki dla teachera.
describe_lengths("Teacher", teacher_lengths)
# Pokazujemy statystyki dla studenta.
describe_lengths("Student", student_lengths)

# Definiujemy kandydatow max_length do porownania.
candidates = [64, 96, 128, 160, 192, 256]

# Definiujemy funkcje liczenia procentu ucietych rekordow.
def truncation_rate(lengths, max_len):
    # Zliczamy rekordy dluzsze niz max_len.
    cut = sum(1 for x in lengths if x > max_len)
    # Zwracamy procent ucietych rekordow.
    return 100.0 * cut / len(lengths)

# Wypisujemy jak duzo danych bedzie uciete dla roznych max_length.
print("\nTruncation rate (%):")
# Iterujemy po kandydatach max_length.
for c in candidates:
    # Liczymy procent ucietych dla teachera.
    t_rate = truncation_rate(teacher_lengths, c)
    # Liczymy procent ucietych dla studenta.
    s_rate = truncation_rate(student_lengths, c)
    # Wypisujemy podsumowanie dla danego kandydata.
    print(f"max_length={c}: teacher={t_rate:.2f}% | student={s_rate:.2f}%")

# Pobieramy limit architektury tokenizera teachera.
teacher_model_max = int(getattr(teacher_tokenizer, "model_max_length", 512))
# Pobieramy limit architektury tokenizera studenta.
student_model_max = int(getattr(student_tokenizer, "model_max_length", 512))

# Korygujemy nienaturalnie duze wartosci model_max_length na praktyczne 512.
if teacher_model_max > 10000:
    # Ustawiamy bezpieczny limit dla teachera.
    teacher_model_max = 512
# Korygujemy nienaturalnie duze wartosci model_max_length na praktyczne 512.
if student_model_max > 10000:
    # Ustawiamy bezpieczny limit dla studenta.
    student_model_max = 512

# Wypisujemy limity wynikajace z architektury modelu.
print("\nModel max length (teacher)=", teacher_model_max)
# Wypisujemy limity wynikajace z architektury modelu.
print("Model max length (student)=", student_model_max)

# Liczymy rekomendacje na podstawie 95 percentyla dla studenta.
recommended_by_data = int(np.percentile(np.array(student_lengths), 95))
# Ograniczamy rekomendacje do limitu modelu i sensownego zakresu CPU.
recommended_max_length = min(recommended_by_data, student_model_max, 256)
# Zapewniamy minimalna sensowna wartosc.
recommended_max_length = max(recommended_max_length, 32)

# Wypisujemy finalna rekomendacje.
print("\nRekomendowany MAX_LENGTH (p95 + CPU cap):", recommended_max_length)
# Wypisujemy obecnie ustawiona wartosc MAX_LENGTH.
print("Aktualny MAX_LENGTH w notebooku:", MAX_LENGTH)
# Podpowiadamy jak zmienic MAX_LENGTH, jesli chcesz.
print("Jesli chcesz uzyc rekomendacji, ustaw: MAX_LENGTH = recommended_max_length")



Teacher token lengths:
min= 4 mean= 30.56 max= 133
p90= 50 p95= 57 p99= 68

Student token lengths:
min= 4 mean= 30.61 max= 150
p90= 50 p95= 57 p99= 68

Truncation rate (%):
max_length=64: teacher=1.96% | student=2.00%
max_length=96: teacher=0.06% | student=0.08%
max_length=128: teacher=0.02% | student=0.02%
max_length=160: teacher=0.00% | student=0.00%
max_length=192: teacher=0.00% | student=0.00%
max_length=256: teacher=0.00% | student=0.00%

Model max length (teacher)= 512
Model max length (student)= 512

Rekomendowany MAX_LENGTH (p95 + CPU cap): 57
Aktualny MAX_LENGTH w notebooku: 128
Jesli chcesz uzyc rekomendacji, ustaw: MAX_LENGTH = recommended_max_length


In [26]:
# Krok 5: ladowanie tokenizerow (fallback, gdy komorka analityczna nie byla uruchomiona)

# Sprawdzamy, czy tokenizer teachera istnieje w aktualnej sesji.
if "teacher_tokenizer" not in globals():
    # Wczytujemy tokenizer teachera, jesli nie istnieje.
    teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_NAME)
# Sprawdzamy, czy tokenizer studenta istnieje w aktualnej sesji.
if "student_tokenizer" not in globals():
    # Wczytujemy tokenizer studenta, jesli nie istnieje.
    student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL_NAME)

# Krok 6: tokenizacja teacher i student

# Tworzymy tokenizacje danych dla teachera.
tokenized_teacher = dataset.map(
    # Definiujemy funkcje tokenizacji batcha dla teachera.
    lambda batch: teacher_tokenizer(
        # Przekazujemy teksty do tokenizera teachera.
        batch["sentence"],
        # Ustawiamy stale dopelnianie do max_length.
        padding="max_length",
        # Ucinamy zbyt dlugie sekwencje.
        truncation=True,
        # Ustawiamy maksymalna dlugosc tokenow.
        max_length=MAX_LENGTH,
    ),
    # Informujemy map, ze funkcja dziala na batchach.
    batched=True,
)

# Tworzymy tokenizacje danych dla studenta.
tokenized_student = dataset.map(
    # Definiujemy funkcje tokenizacji batcha dla studenta.
    lambda batch: student_tokenizer(
        # Przekazujemy teksty do tokenizera studenta.
        batch["sentence"],
        # Ustawiamy stale dopelnianie do max_length.
        padding="max_length",
        # Ucinamy zbyt dlugie sekwencje.
        truncation=True,
        # Ustawiamy maksymalna dlugosc tokenow.
        max_length=MAX_LENGTH,
    ),
    # Informujemy map, ze funkcja dziala na batchach.
    batched=True,
)

# Krok 7: ustawiamy format tensorowy

# Definiujemy kolumny, ktore beda zamienione na tensory.
cols = ["input_ids", "attention_mask", "label", "idx"]
# Ustawiamy format torch dla zbioru teacher.
tokenized_teacher.set_format(type="torch", columns=cols)
# Ustawiamy format torch dla zbioru student.
tokenized_student.set_format(type="torch", columns=cols)

# Krok 8: zapisujemy artefakty na dysk

# Zapisujemy tokenizacje teachera na dysk.
tokenized_teacher.save_to_disk("../artifacts/tokenized_teacher")
# Zapisujemy tokenizacje studenta na dysk.
tokenized_student.save_to_disk("../artifacts/tokenized_student")

# Wyswietlamy potwierdzenie zapisu obu zbiorow.
print("Zapisano tokenized_teacher i tokenized_student")


Map:   0%|          | 0/3876 [00:00<?, ? examples/s]

Map:   0%|          | 0/485 [00:00<?, ? examples/s]

Map:   0%|          | 0/485 [00:00<?, ? examples/s]

Map:   0%|          | 0/3876 [00:00<?, ? examples/s]

Map:   0%|          | 0/485 [00:00<?, ? examples/s]

...

Saving the dataset (0/1 shards):   0%|          | 0/485 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3876 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/485 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/485 [00:00<?, ? examples/s]

Zapisano tokenized_teacher i tokenized_student
